# 03 — Baseline Training

**Goal:** Train a minimal semantic segmentation CNN from scratch to verify the full pipeline runs end-to-end.

---

## What model are we using?

This notebook uses a **very simple 3-layer CNN** as a placeholder.  
Its purpose is to confirm that:
- Data loading works correctly.
- Tensor shapes are correct at every step.
- The loss goes down (even slightly) after one epoch.
- Checkpointing saves and restores weights.

This model will **not** produce good segmentations — that is expected.  
Once the pipeline is verified, replace the model with U-Net, FCN, or DeepLabv3.

> **Prerequisite:** Run notebooks 00–02 first and ensure `pairs` is populated.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

from src.data_paths import RAW_DATA_DIR, MODELS_DIR, ensure_project_dirs
from src.dataset import AI4MarsDataset, find_image_files, find_mask_files, build_pairs_by_stem
from src.train_utils import get_device, save_checkpoint, train_one_epoch, evaluate

ensure_project_dirs()

print(f"PyTorch version: {torch.__version__}")

## Step 1 — Load Pairs

In [ ]:
DATA_ROOT = RAW_DATA_DIR  # adjust if needed

image_files = find_image_files(DATA_ROOT)
mask_files  = find_mask_files(DATA_ROOT)
pairs       = build_pairs_by_stem(image_files, mask_files)

print(f"Total pairs: {len(pairs)}")

if len(pairs) == 0:
    raise RuntimeError(
        "No image/mask pairs found. "
        "Ensure you have downloaded and extracted the AI4Mars data into data/raw/ "
        "and run 01_dataset_inspection.ipynb to verify the structure."
    )

## Step 2 — Hyperparameters

In [ ]:
IMAGE_SIZE   = (256, 256)   # (width, height) — PIL convention
BATCH_SIZE   = 4
NUM_CLASSES  = 4            # soil, bedrock, sand, big_rock
IGNORE_INDEX = 255          # unlabeled pixels in AI4Mars masks
LEARNING_RATE = 1e-3
NUM_EPOCHS   = 1            # set to 1 for a smoke test; increase later
VAL_SPLIT    = 0.1          # 10% of data held out for validation

## Step 3 — Create Train/Val Split and DataLoaders

In [ ]:
# Reproducible split
total = len(pairs)
val_size   = max(1, int(total * VAL_SPLIT))
train_size = total - val_size

# Create the full dataset first so we can split by index
full_dataset = AI4MarsDataset(pairs, image_size=IMAGE_SIZE)
train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples  : {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Step 4 — Verify Batch Shapes

Always check tensor shapes **before** training.  
A shape mismatch will cause a confusing error inside the model.

In [ ]:
images, masks = next(iter(train_loader))
print(f"Image batch shape : {images.shape}   (expected [B, 3, H, W])")
print(f"Mask  batch shape : {masks.shape}    (expected [B, H, W])")
print(f"Image dtype       : {images.dtype}   (expected float32)")
print(f"Mask  dtype       : {masks.dtype}    (expected int64 / long)")
print(f"Image value range : [{images.min():.3f}, {images.max():.3f}]  (expected [0, 1])")

## Step 5 — Define a Placeholder Segmentation Model

This is a **smoke-test model only** — three convolutional layers with no downsampling.
It keeps the spatial resolution fixed and simply maps 3 input channels to `num_classes` output channels.

Replace this with a real architecture (U-Net, FCN, DeepLabv3) once the pipeline is confirmed to work.

In [ ]:
class SimpleSegNet(nn.Module):
    """A minimal 3-layer CNN for segmentation smoke-testing.

    Architecture:
        Conv2d(3  → 16) + ReLU
        Conv2d(16 → 32) + ReLU
        Conv2d(32 → num_classes)

    Input  : [B, 3, H, W]
    Output : [B, num_classes, H, W]  (raw logits, no softmax)
    """

    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            # padding=1 keeps spatial size H x W unchanged
            nn.Conv2d(3,  16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, kernel_size=1),  # 1x1 conv = pixel-wise projection
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# Quick shape check before training
device = get_device()
model = SimpleSegNet(num_classes=NUM_CLASSES).to(device)

dummy = torch.randn(2, 3, *IMAGE_SIZE).to(device)
out   = model(dummy)
print(f"Model output shape: {out.shape}   (expected [2, {NUM_CLASSES}, {IMAGE_SIZE[1]}, {IMAGE_SIZE[0]}])")

## Step 6 — Loss Function and Optimiser

In [ ]:
# CrossEntropyLoss expects:
#   input  : [B, num_classes, H, W]  (raw logits)
#   target : [B, H, W]               (long int class IDs)
# ignore_index=255 skips unlabeled pixels in the loss calculation.
loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Step 7 — Training Loop

In [ ]:
for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 40)

    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_metrics = evaluate(model, val_loader, loss_fn, device,
                           num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX)

    print(f"  Train loss : {train_loss:.4f}")
    print(f"  Val loss   : {val_metrics['val_loss']:.4f}")
    print(f"  Pixel acc  : {val_metrics['pixel_acc']:.4f}")
    print(f"  Mean IoU   : {val_metrics['mean_iou']:.4f}")

## Step 8 — Save Checkpoint

In [ ]:
checkpoint_path = MODELS_DIR / "baseline_epoch01.pth"
save_checkpoint(model, optimizer, epoch=NUM_EPOCHS, path=checkpoint_path)
print(f"Saved checkpoint to {checkpoint_path}")

## Next Steps

- Open `04_evaluation_error_analysis.ipynb` to load the checkpoint and analyse predictions.
- To improve the model, replace `SimpleSegNet` with:
  - **U-Net** — classic encoder-decoder with skip connections.
  - **FCN** — `torchvision.models.segmentation.fcn_resnet50`.
  - **DeepLabv3** — `torchvision.models.segmentation.deeplabv3_resnet50`.